# car2new depth + pointmap demo

Loads a trained F3RM/Nerfstudio run, renders one `IMGPATH`, visualizes depth, and computes camera-coordinate pointmap `(H, W, 3)`.

In [ ]:
from pathlib import Path

import os
# set cuda visible devices
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
# official PyTorch compatibility knob for checkpoints that need weights_only=False
# os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

import matplotlib.pyplot as plt
import numpy as np
import torch
from nerfstudio.utils.eval_utils import eval_setup

CONFIG_PATH = Path("/robodata/smodak/repos/f3rm/mar06_outputs/car2new/f3rm/2026-03-06_164050/config.yml")
DATA_DIR = Path("/robodata/smodak/repos/f3rm/datasets/f3rm/fresh/objaverse/car2new")
IMGPATH = DATA_DIR / "images" / "frame_00057.png"  # pick any image here

assert CONFIG_PATH.exists(), CONFIG_PATH
assert IMGPATH.exists(), IMGPATH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("config:", CONFIG_PATH)
print("img:", IMGPATH)

In [ ]:
config, pipeline, checkpoint_path, step = eval_setup(
    config_path=CONFIG_PATH,
    test_mode="test",
)

print("checkpoint:", checkpoint_path)
print("step:", step)

In [ ]:
img_abs = IMGPATH.resolve()
match = None
for split in ("train", "eval"):
    dataset = getattr(pipeline.datamanager, f"{split}_dataset")
    index_by_path = {Path(p).resolve(): i for i, p in enumerate(dataset.image_filenames)}
    if img_abs in index_by_path:
        match = (split, dataset, index_by_path[img_abs])
        break

if match is None:
    raise ValueError(f"IMGPATH not found in train/eval sets: {img_abs}")

split, dataset, image_idx = match
camera = dataset.cameras[image_idx:image_idx + 1].to(device)
print("split:", split, "image_idx:", image_idx)

In [ ]:
ray_bundle = camera.generate_rays(camera_indices=0, keep_shape=True).to(device)
outputs = pipeline.model.get_outputs_for_camera_ray_bundle(ray_bundle, render_features=False)

depth_ray = outputs["depth"].squeeze(-1)  # distance along ray
pointmap_world = ray_bundle.origins + ray_bundle.directions * depth_ray.unsqueeze(-1)

c2w = camera.camera_to_worlds[0]  # (3, 4)
R = c2w[:, :3]
t = c2w[:, 3]

# Nerfstudio/OpenGL camera coords: +X right, +Y up, -Z forward
pointmap_cam_gl = (pointmap_world - t) @ R

# Convert to OpenCV camera coords: +X right, +Y down, +Z forward
pointmap_cam = pointmap_cam_gl.clone()
pointmap_cam[..., 1] *= -1.0
pointmap_cam[..., 2] *= -1.0

depthmap = pointmap_cam[..., 2]  # camera-space z-depth

depth_np = depthmap.detach().cpu().numpy()
pointmap_np = pointmap_cam.detach().cpu().numpy()
rgb_np = dataset.get_image_float32(image_idx).cpu().numpy()
has_normals = ("normals" in outputs) and ("pred_normals" in outputs)

print("depthmap shape:", depth_np.shape)
print("pointmap shape:", pointmap_np.shape)
print("depth range:", float(depth_np.min()), float(depth_np.max()))
print("normals available:", has_normals)

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(rgb_np)
plt.title("Input image")
plt.axis("off")

plt.subplot(1, 2, 2)
im = plt.imshow(depth_np, cmap="turbo")
plt.title("Depthmap (camera z)")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
# # Optional: save arrays
# np.save("car2new_depthmap_camz.npy", depth_np.astype(np.float32))
# np.save("car2new_pointmap_cam.npy", pointmap_np.astype(np.float32))
# print("saved: car2new_depthmap_camz.npy, car2new_pointmap_cam.npy")